# EDA и разбиение исходного датасета

Проверяем данные и фиксируем выборки. Заполнение пропусков выполняется внутри pipeline при обучении.


In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Подключаем модули проекта при запуске из папки notebooks.
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "config.ini").exists():
    raise RuntimeError("Запустите ноутбук из папки проекта или notebooks")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preprocessing import DataPreprocessor
from src.datasets import read_raw_dataset

## Загрузка и первичный анализ данных


In [ ]:
processor = DataPreprocessor()
df, audit = read_raw_dataset(processor.raw_data_path)

print("Путь к датасету:", processor.raw_data_path)
print("Размер датасета:", df.shape)
print("SHA-256:", audit["sha256"])

print("\nПервые пять строк:")
display(df.head())

print("\nТипы данных:")
display(df.dtypes.to_frame(name="dtype"))

print("\nПропущенные значения:")
display(df.isna().sum().to_frame(name="missing_count"))

print("\nОписательная статистика:")
display(df.describe())

print("\nРаспределение целевой метки:")
display(df["outcome"].value_counts().to_frame(name="count"))

## Нулевые значения и повторы

Нули пяти измерений обозначают пропуски; ноль беременностей сохраняется. Совпадающие строки без идентификаторов пациентов автоматически не удаляем.


In [ ]:
zero_summary = pd.DataFrame(
    {
        "zero_count": (df == 0).sum(),
        "zero_percent": ((df == 0).mean() * 100).round(2),
    }
)

display(zero_summary)
print("Совпадающие строки:", audit["duplicate_rows"])

## Train, validation и test

Разбиение стратифицировано по outcome. Исходные значения и номера строк сохраняются. Медианы здесь не рассчитываем.


In [ ]:
X_train, X_valid, X_test, y_train, y_valid, y_test = processor.split_data(df)

train_ids = set(X_train.index)
valid_ids = set(X_valid.index)
test_ids = set(X_test.index)

assert train_ids.isdisjoint(valid_ids)
assert train_ids.isdisjoint(test_ids)
assert valid_ids.isdisjoint(test_ids)
assert train_ids | valid_ids | test_ids == set(df.index)

split_summary = pd.DataFrame(
    {
        "rows": [len(X_train), len(X_valid), len(X_test)],
        "positive_fraction": [y_train.mean(), y_valid.mean(), y_test.mean()],
    },
    index=["train", "validation", "test"],
)

display(split_summary)